In [ ]:
import pandas as pd
import numpy as np
import re

# PyTorch + Transformers (BERT)
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, TrainingArguments, Trainer


# TensorFlow / Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    BatchNormalization
)

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report, precision_score, recall_score, f1_score, precision_recall_fscore_support
)
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from datasets import Dataset
import json
import joblib
import os

c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("cleaned_data.csv")
df

,QuestionText,Category,Answer
0,"['أيهما', 'أفضل', 'الدراسة', 'السابق', 'الوقت']",التعليم,"['الدراسة', 'الوقت', 'الحالي', 'تعتبر', 'أفضل'..."
1,"['أليس', 'القطن', 'عماد', 'الثروة']",الاقتصاد والعمل,"['القطن', 'يعتبر', 'أهم', 'المنتجات', 'الزراعي..."
2,"['أتصعد', 'الشمس']",التعليم,"['الشمس', 'تصعد', 'الشرق']"
3,"['أتعرف', 'البكتيريا', 'بأنها', 'كائنات', 'حية']",التعليم,"['البكتيريا', 'تعرف', 'بأنها', 'كائنات', 'حية'..."
4,"['أيتكون', 'الهواء', 'أساسا']",التعليم,"['الهواء', 'يتكون', 'أساسا', 'النيتروجين']"
...,...,...,...
5004,"['يحدث', 'تسخين']",العلوم,"['تسخين', 'تتسارع', 'حركة', 'جزيئاته', 'يؤدي',..."
5005,"['الهدف', 'اتفاقية', 'التجارة', 'الحرة', 'لأمر...",الاقتصاد والعمل,"['الهدف', 'اتفاقية', 'التجارة', 'الحرة', 'لأمر..."
5006,"['الغرض', 'اتفاقية', 'التجارة', 'الحرة', 'لأمر...",الاقتصاد والعمل,"['الغرض', 'اتفاقية', 'التجارة', 'الحرة', 'لأمر..."
5007,"['هدف', 'إعجاز', 'القرآن', 'وفقا', 'للمعتقد']",الدين,"['يؤدي', 'الإعجاز', 'غرضين', 'رئيسين', 'الأول'..."


In [4]:
df['QuestionText'] = ( df['QuestionText'] .str.replace("[", "", regex=False) .str.replace("]", "", regex=False) .str.replace("'", "", regex=False) .str.replace(",", " ", regex=False))

In [5]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df['Category'])

In [ ]:
for number, category in enumerate(label_encoder.classes_):
    print(number, "=", category)

with open("label_classes.json", "w", encoding="utf-8") as f:
    json.dump(label_encoder.classes_.tolist(), f, ensure_ascii=False)

0 = الاقتصاد والعمل
1 = البيئة والطاقة
2 = البيولوجيا
3 = التاريخ
4 = الترفيه
5 = التطوع
6 = التعليم
7 = التكنولوجيا
8 = الثقافة
9 = الجغرافيا
10 = الدين
11 = الرياضة
12 = السفر والسياحة
13 = السياسة والقانون
14 = الصحة
15 = العلوم
16 = علم الاجتماع


In [6]:
df

,QuestionText,Category,Answer,label
0,أيهما أفضل الدراسة السابق الوقت,التعليم,"['الدراسة', 'الوقت', 'الحالي', 'تعتبر', 'أفضل'...",6
1,أليس القطن عماد الثروة,الاقتصاد والعمل,"['القطن', 'يعتبر', 'أهم', 'المنتجات', 'الزراعي...",0
2,أتصعد الشمس,التعليم,"['الشمس', 'تصعد', 'الشرق']",6
3,أتعرف البكتيريا بأنها كائنات حية,التعليم,"['البكتيريا', 'تعرف', 'بأنها', 'كائنات', 'حية'...",6
4,أيتكون الهواء أساسا,التعليم,"['الهواء', 'يتكون', 'أساسا', 'النيتروجين']",6
...,...,...,...,...
5004,يحدث تسخين,العلوم,"['تسخين', 'تتسارع', 'حركة', 'جزيئاته', 'يؤدي',...",15
5005,الهدف اتفاقية التجارة الحرة لأمريكا,الاقتصاد والعمل,"['الهدف', 'اتفاقية', 'التجارة', 'الحرة', 'لأمر...",0
5006,الغرض اتفاقية التجارة الحرة لأمريكا,الاقتصاد والعمل,"['الغرض', 'اتفاقية', 'التجارة', 'الحرة', 'لأمر...",0
5007,هدف إعجاز القرآن وفقا للمعتقد,الدين,"['يؤدي', 'الإعجاز', 'غرضين', 'رئيسين', 'الأول'...",10


**BERT model**

In [45]:
X = df['QuestionText']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [46]:
model_name = "aubmindlab/bert-base-arabertv02"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [47]:
train_encodings = tokenizer(
    X_train.tolist(),
    padding=True,
    truncation=True,
    max_length=128
)

test_encodings = tokenizer(
    X_test.tolist(),
    padding=True,
    truncation=True,
    max_length=128
)

In [48]:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": y_train.tolist()
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": y_test.tolist()
})

In [49]:
num_labels = len(label_encoder.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5857.84it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing fro

In [50]:
training_args = TrainingArguments(
    output_dir="./arabert_combined_results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    optim="adamw_torch",
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [51]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    predictions = np.argmax(logits, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )
    
    accuracy = accuracy_score(labels, predictions)
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [52]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [53]:
trainer.train()

c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.144533,1.077682,0.698603,0.705372,0.698603,0.688295
2,0.935063,0.912471,0.738523,0.733518,0.738523,0.734219
3,0.582757,0.911330,0.744511,0.739953,0.744511,0.740132


c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.09s/it]
c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

TrainOutput(global_step=1503, training_loss=1.093811760246952, metrics={'train_runtime': 1119.722, 'train_samples_per_second': 10.736, 'train_steps_per_second': 1.342, 'total_flos': 148278935946960.0, 'train_loss': 1.093811760246952, 'epoch': 3.0})

In [54]:
bert_results = trainer.evaluate()

bert_results

c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.582757,0.911330,3,0.744511,0.739953,0.744511,0.740132


{'eval_loss': 0.9113300442695618,
 'eval_accuracy': 0.7445109780439122,
 'eval_precision': 0.7399531412009778,
 'eval_recall': 0.7445109780439122,
 'eval_f1': 0.7401323429102127}

In [55]:
predictions_output = trainer.predict(test_dataset)

logits = predictions_output.predictions
y_pred = np.argmax(logits, axis=1)

y_true = y_test.tolist()

c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
# Save in the same current folder as the notebook/kernel
save_path = "./arabert_classification_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

joblib.dump(label_encoder, "./arabert_label_encoder.pkl")

print("Saved in:", os.getcwd())
print("Model folder:", save_path)
print("Label encoder:", "./arabert_label_encoder.pkl")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.18it/s]

Saved in: c:\Users\osamo\OneDrive - AL-Hussien bin Abdullah Technical University\NLP\Final
Model folder: ./arabert_classification_model
Label encoder: ./arabert_label_encoder.pkl


**TF-IDF**

In [25]:
# Initialize TF-IDF vectorizer 
vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")

# Fit and transform
tfidf_matrix = vectorizer.fit_transform(df['QuestionText'])

# Convert to DataFrame for readability
tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

# Show TF-IDF values
tfidf

,1,10,100,1000,1048,11,12,120,15,150,...,يواجه,يواجهها,يوازي,يوجد,يوسف,يوضع,يوم,يوما,يوميا,ﷺ
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5006,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5007,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
X = tfidf
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

SVM

In [27]:
# Define model
svm_model = SVC(kernel='linear')

# Train
svm_model.fit(X_train, y_train)

# Predict
y_pred_svm = svm_model.predict(X_test)

In [28]:
# Accuracy
acc_svm = accuracy_score(y_test, y_pred_svm)

# Precision
precision_svm = precision_score(y_test, y_pred_svm, average='weighted')

# Recall
recall_svm = recall_score(y_test, y_pred_svm, average='weighted')

# F1-score
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')

print(f"SVM Accuracy:  {acc_svm:.4f}")
print(f"SVM Precision: {precision_svm:.4f}")
print(f"SVM Recall:    {recall_svm:.4f}")
print(f"SVM F1-score:  {f1_svm:.4f}")

SVM Accuracy:  0.6587
SVM Precision: 0.6808
SVM Recall:    0.6587
SVM F1-score:  0.6554


c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Logistic Regression

In [29]:
# Define model
lr_model = LogisticRegression(max_iter=1000)

# Train
lr_model.fit(X_train, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test)

In [30]:
# Accuracy
acc_lr = accuracy_score(y_test, y_pred_lr)

# Precision
precision_lr = precision_score(y_test, y_pred_lr, average='weighted')

# Recall
recall_lr = recall_score(y_test, y_pred_lr, average='weighted')

# F1-score
f1_lr = f1_score(y_test, y_pred_lr, average='weighted')

print(f"LR Accuracy:  {acc_lr:.4f}")
print(f"LR Precision: {precision_lr:.4f}")
print(f"LR Recall:    {recall_lr:.4f}")
print(f"LR F1-score:  {f1_lr:.4f}")

LR Accuracy:  0.6307
LR Precision: 0.6720
LR Recall:    0.6307
LR F1-score:  0.6225


c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


**BOW (Unigram)**

In [31]:
# Initialize CountVectorizer for unigram BoW
cv_uni_vec = CountVectorizer(
    max_features=600,
    stop_words=None,
    ngram_range=(1, 1),
    token_pattern=r"(?u)\b\w+\b"
)

cv_uni = cv_uni_vec.fit_transform(df['QuestionText'])

# Convert to DataFrame for readability
cv_uni_df = pd.DataFrame(
    cv_uni.toarray(),
    columns=cv_uni_vec.get_feature_names_out()
)

# Show BoW unigram values
cv_uni_df

,1,100,2,20,3,30,أبي,أثناء,أساسيا,أسباب,...,يعيش,يفضل,يقاس,يقصد,يقطعها,يقع,يكون,يمكن,ينصح,يوجد
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5005,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5006,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5007,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
X = cv_uni
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

SVM

In [33]:
# Define model
svm_model = SVC(kernel='linear')

# Train
svm_model.fit(X_train, y_train)

# Predict
y_pred_svm = svm_model.predict(X_test)

In [34]:
# Accuracy
acc_svm = accuracy_score(y_test, y_pred_svm)

# Precision
precision_svm = precision_score(y_test, y_pred_svm, average='weighted')

# Recall
recall_svm = recall_score(y_test, y_pred_svm, average='weighted')

# F1-score
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')

print(f"SVM Accuracy:  {acc_svm:.4f}")
print(f"SVM Precision: {precision_svm:.4f}")
print(f"SVM Recall:    {recall_svm:.4f}")
print(f"SVM F1-score:  {f1_svm:.4f}")

SVM Accuracy:  0.5898
SVM Precision: 0.6091
SVM Recall:    0.5898
SVM F1-score:  0.5901


c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Logistic Regression

In [35]:
# Define model
lr_model = LogisticRegression(max_iter=1000)

# Train
lr_model.fit(X_train, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test)

In [36]:
# Accuracy
acc_lr = accuracy_score(y_test, y_pred_lr)

# Precision
precision_lr = precision_score(y_test, y_pred_lr, average='weighted')

# Recall
recall_lr = recall_score(y_test, y_pred_lr, average='weighted')

# F1-score
f1_lr = f1_score(y_test, y_pred_lr, average='weighted')

print(f"LR Accuracy:  {acc_lr:.4f}")
print(f"LR Precision: {precision_lr:.4f}")
print(f"LR Recall:    {recall_lr:.4f}")
print(f"LR F1-score:  {f1_lr:.4f}")

LR Accuracy:  0.6088
LR Precision: 0.6378
LR Recall:    0.6088
LR F1-score:  0.6098


c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


**Bert Embeddings**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")
model_bert = AutoModel.from_pretrained("aubmindlab/bert-base-arabertv02")

def get_bert_embeddings(texts):
    inputs = tokenizer(
        texts.tolist(),
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model_bert(**inputs) #Put tokens in Bert without training gradients

    embeddings = outputs.last_hidden_state.mean(dim=1) #Return embeddings from the last hidden state
    return embeddings.cpu().numpy()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2782.70it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [38]:
X_text = df['QuestionText']
y = df['label']

X_bert = get_bert_embeddings(X_text)

In [39]:
np.save("X_bert_embeddings.npy", X_bert)
np.save("y_labels.npy", y.values)

In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bert,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

SVM

In [41]:
# Define model
svm_model = SVC(kernel='linear')

# Train
svm_model.fit(X_train, y_train)

# Predict
y_pred_svm = svm_model.predict(X_test)

In [42]:
# Accuracy
acc_svm = accuracy_score(y_test, y_pred_svm)

# Precision
precision_svm = precision_score(y_test, y_pred_svm, average='weighted')

# Recall
recall_svm = recall_score(y_test, y_pred_svm, average='weighted')

# F1-score
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')

print(f"SVM Accuracy:  {acc_svm:.4f}")
print(f"SVM Precision: {precision_svm:.4f}")
print(f"SVM Recall:    {recall_svm:.4f}")
print(f"SVM F1-score:  {f1_svm:.4f}")

SVM Accuracy:  0.7146
SVM Precision: 0.7169
SVM Recall:    0.7146
SVM F1-score:  0.7122


c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Logistic Regression

In [43]:
# Define model
lr_model = LogisticRegression(max_iter=1000)

# Train
lr_model.fit(X_train, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test)

In [44]:
# Accuracy
acc_lr = accuracy_score(y_test, y_pred_lr)

# Precision
precision_lr = precision_score(y_test, y_pred_lr, average='weighted')

# Recall
recall_lr = recall_score(y_test, y_pred_lr, average='weighted')

# F1-score
f1_lr = f1_score(y_test, y_pred_lr, average='weighted')

print(f"LR Accuracy:  {acc_lr:.4f}")
print(f"LR Precision: {precision_lr:.4f}")
print(f"LR Recall:    {recall_lr:.4f}")
print(f"LR F1-score:  {f1_lr:.4f}")

LR Accuracy:  0.6986
LR Precision: 0.7009
LR Recall:    0.6986
LR F1-score:  0.6963


c:\Users\osamo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
